In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install spectral
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

In [7]:
# ================================
# Salinas HSI (Memory-Safe, Same Architecture/Hyperparams)
# ================================
# !pip install spectral

import os, time, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, accuracy_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
import spectral
import gc, torch
gc.collect()
tf.keras.backend.clear_session()

# --------- Config ---------
dataset = 'Ho'                 # Salinas
windowSize = 25                # keep as requested
K = 15                         # PCA components
train_ratio = 0.10             # 10% per class for training
batch_size = 256               # keep same as your code
epochs = 100                   # keep same
model_name = "Ho_PCA_15_Light_perc_10_Optimized"
results_folder = "results"
os.makedirs(results_folder, exist_ok=True)

# Optional mixed precision to save memory
mixed_precision.set_global_policy("mixed_float16")
tf.keras.backend.clear_session()
gc.collect()

# --------- Data Loading ---------
def loadData(name):
    # Adjust base path if your files are elsewhere
    base = "/content/drive/MyDrive/Colab Notebooks/dataset/"  # e.g., "/content/drive/MyDrive/Colab Notebooks/dataset/"
    if name == 'Ho':
        data   = sio.loadmat(os.path.join(base, 'Houston.mat'))['houston']
        labels = sio.loadmat(os.path.join(base, 'Houston_gt.mat'))['houston_gt']
    else:
        raise ValueError("Only 'Ho' (Houston) implemented in this script.")
    return data, labels

def applyPCA(X, numComponents):
    Xr = X.reshape(-1, X.shape[2]).astype(np.float32)
    pca = PCA(n_components=numComponents, whiten=True)
    Xp = pca.fit_transform(Xr)
    return Xp.reshape(X.shape[0], X.shape[1], numComponents), pca

def padWithZeros(X, margin=0):
    return np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')

# --------- Memory-Safe Patch Generator ---------
class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, coords, labels, full_cube, patch_size=25,
                 batch_size=256, shuffle=True, n_classes=20):
        self.coords = coords
        self.labels = labels
        self.full_cube = full_cube
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.half = patch_size // 2
        self.indices = np.arange(len(labels))
        self.padded = padWithZeros(full_cube, self.half)  # pad once
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        # preallocate for speed
        batch_X = np.empty((len(idxs), self.patch_size, self.patch_size, self.full_cube.shape[2], 1), dtype=np.float32)
        batch_y = np.empty((len(idxs),), dtype=np.int32)
        for m, k in enumerate(idxs):
            r, c = self.coords[k]
            patch = self.padded[r:r+self.patch_size, c:c+self.patch_size, :]
            batch_X[m, ..., 0] = patch
            batch_y[m] = self.labels[k]
        # NOTE: to_categorical returns float; mixed precision will cast as needed
        batch_y = to_categorical(batch_y, num_classes=self.n_classes)
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# --------- Per-Class Split on Coordinates ---------
def splitTrainTestCoords(coords, labels, train_ratio=0.1, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []
    for cl in np.unique(labels):
        idx = np.where(labels == cl)[0]
        np.random.shuffle(idx)
        n_train = max(1, int(len(idx) * train_ratio))
        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])
    return coords[train_idx], coords[test_idx], labels[train_idx], labels[test_idx]

# --------- Build Model (EXACT architecture you used) ---------
def build_model(S, L, num_classes):
    inp = tf.keras.layers.Input((S, S, L, 1))
    x = tf.keras.layers.Conv3D(filters=8,  kernel_size=(3,3,7), activation='relu')(inp)
    x = tf.keras.layers.Conv3D(filters=16, kernel_size=(3,3,5), activation='relu')(x)
    x = tf.keras.layers.Conv3D(filters=32, kernel_size=(3,3,3), activation='relu')(x)
    # reshape to 2D convs
    conv3d_shape = x.shape  # (None, H, W, D, C)
    x = tf.keras.layers.Reshape((conv3d_shape[1], conv3d_shape[2],
                                 conv3d_shape[3]*conv3d_shape[4]))(x)
    x = tf.keras.layers.Conv2D(filters=64,  kernel_size=(3,3), activation='relu')(x)
    x = tf.keras.layers.Flatten()(x)
    #x = tf.keras.layers.Conv2D(filters=96,  kernel_size=(3,3), activation='relu')(x)
    #x = tf.keras.layers.MaxPooling2D(pool_size=(2,2))(x)
    #x = tf.keras.layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu')(x)
    #x = tf.keras.layers.GlobalAveragePooling2D()(x)
    #x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dense(units=256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.models.Model(inputs=inp, outputs=out)
    # Adam 0.001 as in your code
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

# =========================
# Pipeline
# =========================
# 1) Load + PCA
X_full, y_full = loadData(dataset)
n_classes = int(y_full.max())  # 16 for Salinas
X_pca, pca = applyPCA(X_full, numComponents=K)

# 2) Build coordinate list (only labeled pixels with border safety)
margin = windowSize // 2
coords, labels = [], []
for r in range(margin, X_pca.shape[0]-margin):
    for c in range(margin, X_pca.shape[1]-margin):
        lab = y_full[r, c]
        if lab > 0:
            coords.append((r, c))
            labels.append(lab - 1)  # zero-based
coords = np.array(coords, dtype=np.int32)
labels = np.array(labels, dtype=np.int32)

# 3) Train/Test split (10% per class)
train_coords, test_coords, ytrain_idx, ytest_idx = splitTrainTestCoords(coords, labels, train_ratio=train_ratio)

# 4) Generators
train_gen = PatchGenerator(train_coords, ytrain_idx, X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=True, n_classes=n_classes)
test_gen  = PatchGenerator(test_coords,  ytest_idx,  X_pca, patch_size=windowSize,
                           batch_size=batch_size, shuffle=False, n_classes=n_classes)

# 5) Model
model = build_model(windowSize, K, n_classes)

# Save model summary
with open(os.path.join(results_folder, f"{model_name}_summary.txt"), "w") as f:
    model.summary(print_fn=lambda s: f.write(s + "\n"))

# 6) Train
tic = time.perf_counter()
history = model.fit(train_gen, validation_data=test_gen, epochs=epochs, verbose=2)
toc = time.perf_counter()
train_time = toc - tic

# 7) Plots: accuracy & loss
plt.figure()
plt.plot(history.history['accuracy'], label="Train Acc")
plt.plot(history.history['val_accuracy'], label="Val Acc")
plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Val Loss")
plt.xlabel("Epochs"); plt.ylabel("Value"); plt.legend()
plt.title("Training vs Validation")
plt.savefig(os.path.join(results_folder, f"{model_name}_training.png"), dpi=150)
plt.close()

# 8) Evaluation on Test Set
tic1 = time.perf_counter()
y_pred_prob = model.predict(test_gen, verbose=0)
toc1 = time.perf_counter()
test_time = toc1 - tic1

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = ytest_idx  # already zero-based labels

# Metrics
classification = classification_report(y_true, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
oa = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
each_acc = np.nan_to_num(np.diag(cm) / cm.sum(axis=1, keepdims=False))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(y_true, y_pred)

# Confusion matrix figure
plt.figure(figsize=(8,6))
plt.imshow(cm, interpolation='nearest', cmap='viridis')
plt.title('Confusion Matrix'); plt.colorbar()
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(results_folder, f"{model_name}_confusion_matrix.png"), dpi=150)
plt.close()

# 9) Save Results Text (all metrics + model desc)
with open(os.path.join(results_folder, f"{model_name}_results.txt"), "w") as f:
    f.write(f"Model: 3D→2D CNN (Conv3D(8,3x3x7)->Conv3D(16,3x3x5)->Conv3D(32,3x3x3)"
            f"->Reshape->Conv2D(24,3x3)->Conv2D(96,3x3)->MaxPool2D(2x2)->Conv2D(128,3x3)"
            f"->GAP->Dense(128)->Dropout(0.4)->Dense(64)->Dropout(0.4)->Dense({n_classes}))\n")
    f.write(f"Optimizer: Adam(1e-3), Batch Size: {batch_size}, Epochs: {epochs}\n")
    f.write(f"Window Size: {windowSize}x{windowSize}, PCA Components: {K}\n")
    f.write(f"Training ratio per class: {int(train_ratio*100)}%\n\n")

    f.write(f"Training time: {train_time:.2f} s\n")
    f.write(f"Testing time: {test_time:.2f} s\n\n")

    f.write(f"Overall Accuracy: {oa*100:.2f}%\n")
    f.write(f"Average Accuracy: {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision (weighted): {precision*100:.2f}%\n")
    f.write(f"Recall (weighted): {recall*100:.2f}%\n")
    f.write(f"F1-score (weighted): {f1*100:.2f}%\n\n")

    f.write("Classwise Accuracy (%):\n")
    f.write(", ".join([f"{x*100:.2f}" for x in each_acc]) + "\n\n")

    f.write("Classification Report:\n")
    f.write(classification + "\n\n")

    f.write("Confusion Matrix:\n")
    f.write(np.array2string(cm) + "\n")

# 10) Full Map Prediction (batched, memory-safe)
print("Predicting full map (batched)...")
PATCH = windowSize
pad = PATCH // 2
Xp = padWithZeros(X_pca, pad)
H, W = y_full.shape
outputs = np.zeros((H, W), dtype=np.int32)

# Collect coords of labeled pixels
coords_all = [(r, c) for r in range(pad, H-pad) for c in range(pad, W-pad) if y_full[r, c] > 0]
B = 2048  # batch for full-map inference
for start in range(0, len(coords_all), B):
    batch_coords = coords_all[start:start+B]
    batch = np.empty((len(batch_coords), PATCH, PATCH, K, 1), dtype=np.float32)
    for i, (r, c) in enumerate(batch_coords):
        patch = Xp[r-pad:r+pad+1, c-pad:c+pad+1, :]
        batch[i, ..., 0] = patch
    preds = np.argmax(model.predict(batch, verbose=0), axis=1)
    for (r, c), p in zip(batch_coords, preds):
        outputs[r, c] = p + 1  # back to 1..C

# Save classified and ground-truth maps
spectral.save_rgb(os.path.join(results_folder, f"{model_name}_classified_map.jpg"),
                  outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f"{model_name}_groundtruth.jpg"),
                  y_full, colors=spectral.spy_colors)

# 11) Zip everything
zip_path = f"{model_name}_outputs.zip"
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for fn in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, fn), arcname=fn)

print(f"✅ Done. All outputs saved in: {zip_path}")


Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


6/6 - 20s - 3s/step - accuracy: 0.1737 - loss: 2.5768 - val_accuracy: 0.3686 - val_loss: 2.0489
Epoch 2/100
6/6 - 4s - 738ms/step - accuracy: 0.4010 - loss: 1.8780 - val_accuracy: 0.6106 - val_loss: 1.3760
Epoch 3/100
6/6 - 2s - 348ms/step - accuracy: 0.5796 - loss: 1.3513 - val_accuracy: 0.7332 - val_loss: 0.9732
Epoch 4/100
6/6 - 2s - 288ms/step - accuracy: 0.6692 - loss: 1.0679 - val_accuracy: 0.7928 - val_loss: 0.7029
Epoch 5/100
6/6 - 2s - 288ms/step - accuracy: 0.7491 - loss: 0.7457 - val_accuracy: 0.8765 - val_loss: 0.4965
Epoch 6/100
6/6 - 2s - 348ms/step - accuracy: 0.8353 - loss: 0.5513 - val_accuracy: 0.9059 - val_loss: 0.3665
Epoch 7/100
6/6 - 2s - 365ms/step - accuracy: 0.8860 - loss: 0.4182 - val_accuracy: 0.9273 - val_loss: 0.2949
Epoch 8/100
6/6 - 2s - 346ms/step - accuracy: 0.9083 - loss: 0.3134 - val_accuracy: 0.9407 - val_loss: 0.2372
Epoch 9/100
6/6 - 2s - 288ms/step - accuracy: 0.9291 - loss: 0.2330 - val_accuracy: 0.9480 - val_loss: 0.2051
Epoch 10/100
6/6 - 2s - 